# AutoDDG baseline on iNaturalist — all experiment models

Run AutoDDG on every species CSV in the bundled iNaturalist dataset for all five LLMs used by the M³ experiments. This is one resumable notebook: use **Run All** once.

For each model, the notebook writes one JSON result per species under `outputs/autoddg_inat_<model_slug>/`. Each CSV/model pair has a 10-minute wall-clock limit. Valid successful outputs and explicit timeout records are skipped on resume; ordinary failures remain retryable.

AutoDDG produces free-text dataset descriptions plus structural and semantic profiles. It does not produce the Croissant-only `spatialCoverage` / `temporalCoverage` output used by the M³ iNaturalist pipeline.

In [1]:
import json
import logging
import os
import re
import sys
import time
import traceback
from pathlib import Path

from dotenv import load_dotenv

BASE = Path('../..').resolve()
sys.path.insert(0, str(BASE))

load_dotenv(BASE / '.env')
os.environ['LLM_PROVIDER'] = 'surf'

from src.baseline.autoddg.csv_runner import describe_csv_with_timeout
from src.config import LLM_PROVIDER

# The five models used by the repository's iNaturalist/PANGAEA experiments.
MODEL_NAMES = [
    'Sehyo/Qwen3.5-122B-A10B-NVFP4',
    'mistralai/Mistral-Small-3.2-24B-Instruct-2506',
    'Qwen/Qwen3.6-35B-A3B-FP8',
    'default-text-large',
    'mistralai/Devstral-Small-2-24B-Instruct-2512',
]

SAMPLE_SIZE = 20
DESCRIPTION_WORDS = 100
RANDOM_STATE = 9
PROCESS_TIMEOUT_SECONDS = 600

INATURALIST_DIR_CANDIDATES = [
    BASE / 'data/experiment/inaturalist',
    BASE / 'data/experiment/inaturalist_100_species_1000_obs/inaturalist_100_species_1000_obs',
]


def resolve_inaturalist_dir() -> Path:
    for path in INATURALIST_DIR_CANDIDATES:
        if path.is_dir():
            return path
    searched = '\n  '.join(str(p) for p in INATURALIST_DIR_CANDIDATES)
    raise FileNotFoundError(f'iNaturalist data directory not found. Tried:\n  {searched}')


def model_slug(model_name: str) -> str:
    return re.sub(r'[^\w.\-]+', '_', model_name.replace('/', '_'))


def output_dir_for(model_name: str) -> Path:
    return BASE / 'outputs' / f'autoddg_inat_{model_slug(model_name)}'


def run_log_for(model_name: str) -> Path:
    return output_dir_for(model_name) / 'run_log.jsonl'


INATURALIST_DIR = resolve_inaturalist_dir()
all_csv_files = sorted(INATURALIST_DIR.glob('*.csv'))
if not all_csv_files:
    raise FileNotFoundError(f'No CSV files found in {INATURALIST_DIR}')

for model_name in MODEL_NAMES:
    output_dir_for(model_name).mkdir(parents=True, exist_ok=True)

logger = logging.getLogger()
logger.setLevel(logging.INFO)
for handler in logger.handlers[:]:
    logger.removeHandler(handler)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s'))
logger.addHandler(handler)

print(f'LLM provider: {LLM_PROVIDER}')
print(f'Models: {len(MODEL_NAMES)}')
for model_name in MODEL_NAMES:
    print(f'  - {model_name}')
print(f'Per-file/model timeout: {PROCESS_TIMEOUT_SECONDS}s')
print(f'Data directory: {INATURALIST_DIR}')
print(f'iNaturalist CSV files: {len(all_csv_files)}')
print(f'Total model-file jobs: {len(MODEL_NAMES) * len(all_csv_files)}')

LLM provider: surf
Models: 5
  - Sehyo/Qwen3.5-122B-A10B-NVFP4
  - mistralai/Mistral-Small-3.2-24B-Instruct-2506
  - Qwen/Qwen3.6-35B-A3B-FP8
  - default-text-large
  - mistralai/Devstral-Small-2-24B-Instruct-2512
Per-file/model timeout: 600s
Data directory: /home/com3dian/Github/metadata_agent/data/experiment/inaturalist_100_species_1000_obs/inaturalist_100_species_1000_obs
iNaturalist CSV files: 100
Total model-file jobs: 500


In [2]:
def output_path_for(csv_path: Path, model_name: str) -> Path:
    return output_dir_for(model_name) / f'{csv_path.stem}_autoddg.json'


def skip_path_for(csv_path: Path, model_name: str) -> Path:
    return output_dir_for(model_name) / f'{csv_path.stem}_autoddg.skipped.json'


def load_json_if_valid(path: Path):
    if not path.is_file():
        return None
    try:
        with open(path, encoding='utf-8') as f:
            value = json.load(f)
    except (OSError, json.JSONDecodeError):
        return None
    return value if isinstance(value, dict) else None


def is_valid_success(csv_path: Path, model_name: str) -> bool:
    record = load_json_if_valid(output_path_for(csv_path, model_name))
    return bool(
        record
        and record.get('status') == 'ok'
        and record.get('csv') == csv_path.name
        and record.get('model') == model_name
        and isinstance(record.get('description'), str)
        and record['description'].strip()
    )


def is_valid_timeout(csv_path: Path, model_name: str) -> bool:
    record = load_json_if_valid(skip_path_for(csv_path, model_name))
    return bool(
        record
        and record.get('status') == 'skipped'
        and record.get('reason') == 'timeout'
        and record.get('csv') == csv_path.name
        and record.get('model') == model_name
    )


def is_processed(csv_path: Path, model_name: str) -> bool:
    return is_valid_success(csv_path, model_name) or is_valid_timeout(csv_path, model_name)


def append_run_log(model_name: str, event: dict) -> None:
    payload = {
        'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S'),
        **event,
    }
    with open(run_log_for(model_name), 'a', encoding='utf-8') as f:
        f.write(json.dumps(payload, ensure_ascii=False) + '\n')


def write_timeout_skip(csv_path: Path, model_name: str, message: str) -> tuple[Path, dict]:
    record = {
        'status': 'skipped',
        'reason': 'timeout',
        'csv': csv_path.name,
        'dataset_id': csv_path.stem,
        'provider': LLM_PROVIDER,
        'model': model_name,
        'timeout_seconds': PROCESS_TIMEOUT_SECONDS,
        'runtime_seconds': PROCESS_TIMEOUT_SECONDS,
        'message': message,
    }
    skip_file = skip_path_for(csv_path, model_name)
    with open(skip_file, 'w', encoding='utf-8') as f:
        json.dump(record, f, indent=2, ensure_ascii=False)
    append_run_log(model_name, {**record, 'output_file': str(skip_file)})
    return skip_file, record


def save_success(csv_path: Path, model_name: str, result: dict) -> Path:
    output_file = output_path_for(csv_path, model_name)
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    append_run_log(model_name, {
        'status': 'ok',
        'csv': csv_path.name,
        'dataset_id': csv_path.stem,
        'provider': LLM_PROVIDER,
        'model': model_name,
        'runtime_seconds': result.get('runtime_seconds'),
        'output_file': str(output_file),
    })
    return output_file


def run_autoddg(csv_path: Path, model_name: str) -> dict:
    try:
        return describe_csv_with_timeout(
            csv_path,
            timeout_seconds=PROCESS_TIMEOUT_SECONDS,
            sample_size=SAMPLE_SIZE,
            description_words=DESCRIPTION_WORDS,
            random_state=RANDOM_STATE,
            model_name=model_name,
        )
    except TimeoutError as exc:
        skip_file, record = write_timeout_skip(csv_path, model_name, str(exc))
        print(f'  TIMEOUT after {PROCESS_TIMEOUT_SECONDS}s: {csv_path.name}')
        print(f'  Skip record: {skip_file}')
        return record

## Current progress

This cell is read-only with respect to model calls. Run it whenever you want to see how many species remain for each model.

In [3]:
for model_name in MODEL_NAMES:
    completed = sum(is_processed(path, model_name) for path in all_csv_files)
    print(f'{model_name}: {completed}/{len(all_csv_files)} complete; {len(all_csv_files) - completed} remaining')
    print(f'  output: {output_dir_for(model_name)}')

Sehyo/Qwen3.5-122B-A10B-NVFP4: 100/100 complete; 0 remaining
  output: /home/com3dian/Github/metadata_agent/outputs/autoddg_inat_Sehyo_Qwen3.5-122B-A10B-NVFP4
mistralai/Mistral-Small-3.2-24B-Instruct-2506: 100/100 complete; 0 remaining
  output: /home/com3dian/Github/metadata_agent/outputs/autoddg_inat_mistralai_Mistral-Small-3.2-24B-Instruct-2506
Qwen/Qwen3.6-35B-A3B-FP8: 100/100 complete; 0 remaining
  output: /home/com3dian/Github/metadata_agent/outputs/autoddg_inat_Qwen_Qwen3.6-35B-A3B-FP8
default-text-large: 100/100 complete; 0 remaining
  output: /home/com3dian/Github/metadata_agent/outputs/autoddg_inat_default-text-large
mistralai/Devstral-Small-2-24B-Instruct-2512: 53/100 complete; 47 remaining
  output: /home/com3dian/Github/metadata_agent/outputs/autoddg_inat_mistralai_Devstral-Small-2-24B-Instruct-2512


## Run every model on every remaining species

The outer loop selects a model and the inner loop processes its remaining species CSVs. The notebook checkpoints after every CSV, so interruption loses at most the currently running CSV. Rerun this cell—or use **Run All** again—to continue.

In [4]:
all_model_summaries = {}

for model_index, model_name in enumerate(MODEL_NAMES, start=1):
    pending_files = [path for path in all_csv_files if not is_processed(path, model_name)]
    previously_processed = len(all_csv_files) - len(pending_files)

    print(f'\n{"#" * 72}')
    print(f'MODEL {model_index}/{len(MODEL_NAMES)}: {model_name}')
    print(f'Output directory: {output_dir_for(model_name)}')
    print(f'Remaining: {len(pending_files)}; already complete/timed out: {previously_processed}')

    model_summary = {
        'model': model_name,
        'previously_processed': previously_processed,
        'successes_this_run': 0,
        'timeouts_this_run': 0,
        'failures_this_run': 0,
        'results': [],
    }

    for file_index, csv_path in enumerate(pending_files, start=1):
        print(f'\n[{model_index}/{len(MODEL_NAMES)} | {file_index}/{len(pending_files)}] {csv_path.name}')
        entry = {
            'csv': csv_path.name,
            'status': 'error',
            'output_file': None,
            'error': None,
        }

        try:
            result = run_autoddg(csv_path, model_name)
            if result.get('status') == 'skipped':
                model_summary['timeouts_this_run'] += 1
                entry['status'] = 'timeout'
                entry['error'] = result.get('message')
                entry['output_file'] = str(skip_path_for(csv_path, model_name))
                print(f"  SKIPPED (timeout) -> {skip_path_for(csv_path, model_name).name}")
            else:
                output_file = save_success(csv_path, model_name, result)
                model_summary['successes_this_run'] += 1
                entry['status'] = 'ok'
                entry['output_file'] = str(output_file)
                print(
                    f"  topic={result['topic']!r} "
                    f"runtime={result['runtime_seconds']}s -> {output_file.name}"
                )
        except Exception as exc:
            model_summary['failures_this_run'] += 1
            entry['error'] = str(exc)
            append_run_log(model_name, {
                'status': 'error',
                'csv': csv_path.name,
                'dataset_id': csv_path.stem,
                'provider': LLM_PROVIDER,
                'model': model_name,
                'message': str(exc),
            })
            print(f'  FAILED (retryable): {exc}')
            traceback.print_exc()

        model_summary['results'].append(entry)

    model_summary['total_processed_after_run'] = sum(
        is_processed(path, model_name) for path in all_csv_files
    )
    all_model_summaries[model_name] = model_summary

    print(f'\nCompleted model: {model_name}')
    print(f"  successes this run: {model_summary['successes_this_run']}")
    print(f"  timeouts this run: {model_summary['timeouts_this_run']}")
    print(f"  failures still retryable: {model_summary['failures_this_run']}")
    print(f"  total checkpointed: {model_summary['total_processed_after_run']}/{len(all_csv_files)}")

print(f'\n{"=" * 72}')
print('ALL-MODEL SUMMARY')
for model_name, summary in all_model_summaries.items():
    print(
        f"{model_name}: {summary['total_processed_after_run']}/{len(all_csv_files)} checkpointed; "
        f"{summary['failures_this_run']} retryable failure(s) this run"
    )


########################################################################
MODEL 1/5: Sehyo/Qwen3.5-122B-A10B-NVFP4
Output directory: /home/com3dian/Github/metadata_agent/outputs/autoddg_inat_Sehyo_Qwen3.5-122B-A10B-NVFP4
Remaining: 0; already complete/timed out: 100

Completed model: Sehyo/Qwen3.5-122B-A10B-NVFP4
  successes this run: 0
  timeouts this run: 0
  failures still retryable: 0
  total checkpointed: 100/100

########################################################################
MODEL 2/5: mistralai/Mistral-Small-3.2-24B-Instruct-2506
Output directory: /home/com3dian/Github/metadata_agent/outputs/autoddg_inat_mistralai_Mistral-Small-3.2-24B-Instruct-2506
Remaining: 0; already complete/timed out: 100

Completed model: mistralai/Mistral-Small-3.2-24B-Instruct-2506
  successes this run: 0
  timeouts this run: 0
  failures still retryable: 0
  total checkpointed: 100/100

########################################################################
MODEL 3/5: Qwen/Qwen3.6-35B-A3B-FP

/home/com3dian/miniconda3/envs/sam2/lib/python3.12/site-packages/datamart_profiler/core.py:199: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.astype(object).fillna('').astype(str)


  topic='Lomandra obliqua observations' runtime=78.506s -> lomandra_obliqua_autoddg.json

[5/5 | 2/47] lomatium_orientale.csv


/home/com3dian/miniconda3/envs/sam2/lib/python3.12/site-packages/datamart_profiler/core.py:199: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.astype(object).fillna('').astype(str)


  topic='**Lomatium orientale observations**' runtime=77.942s -> lomatium_orientale_autoddg.json

[5/5 | 3/47] luridiblatta_trivittata.csv


...

  topic='**Florida Plants**' runtime=78.857s -> tiedemannia_filiformis_autoddg.json

[5/5 | 46/47] veniliornis_passerinus.csv


/home/com3dian/miniconda3/envs/sam2/lib/python3.12/site-packages/datamart_profiler/core.py:199: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.astype(object).fillna('').astype(str)


  topic='**Passerine Woodpecker**' runtime=78.568s -> veniliornis_passerinus_autoddg.json

[5/5 | 47/47] wisteriopsis_reticulata.csv


/home/com3dian/miniconda3/envs/sam2/lib/python3.12/site-packages/datamart_profiler/core.py:199: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.astype(object).fillna('').astype(str)


  topic='**Wisteriopsis reticulata observations**' runtime=78.743s -> wisteriopsis_reticulata_autoddg.json

Completed model: mistralai/Devstral-Small-2-24B-Instruct-2512
  successes this run: 47
  timeouts this run: 0
  failures still retryable: 0
  total checkpointed: 100/100

ALL-MODEL SUMMARY
Sehyo/Qwen3.5-122B-A10B-NVFP4: 100/100 checkpointed; 0 retryable failure(s) this run
mistralai/Mistral-Small-3.2-24B-Instruct-2506: 100/100 checkpointed; 0 retryable failure(s) this run
Qwen/Qwen3.6-35B-A3B-FP8: 100/100 checkpointed; 0 retryable failure(s) this run
default-text-large: 100/100 checkpointed; 0 retryable failure(s) this run
mistralai/Devstral-Small-2-24B-Instruct-2512: 100/100 checkpointed; 0 retryable failure(s) this run
